# Qwen3-4B Chain-of-Thought baseline

Generate complete reasoning traces for a small deterministic GSM8K development sample. Use a GPU Colab runtime.

In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
os.chdir(REPOSITORY_DIR)
%pip install -q -r requirements.txt

In [ ]:
from src.data.gsm8k import load_gsm8k

gsm8k = load_gsm8k(development_size=10, seed=42)
print({name: len(split) for name, split in gsm8k.items()})

In [ ]:
from src.models.qwen import GenerationSettings, QwenGenerator

settings = GenerationSettings(max_new_tokens=1024, do_sample=False, seed=42)
generator = QwenGenerator.from_pretrained(settings=settings)
print(f"Loaded {generator.model_id} with {generator.precision} precision.")

In [ ]:
from src.baseline.cot import run_cot_baseline

records = run_cot_baseline(gsm8k["development"], generator, limit=3)
assert len(records) == 3
assert all(record["reasoning_trace"].strip() for record in records)
assert all(record["input_tokens"] > 0 for record in records)
assert all(record["output_tokens"] > 0 for record in records)
print("Generated 3 complete baseline records.")

In [ ]:
import json

print(json.dumps(records[0], indent=2, ensure_ascii=False))